# Lab / Practical Work: Spectral Methods and SDP Relaxations for Community Detection
**Course reference:** *Spectral methods and SDP relaxations for community detection in graphs* (G. Lecué)

---

### Objectives
1. **Part 1: Graph Laplacian & Algebraic Connectivity** — Understand the unnormalized Laplacian $L = D - A$, its spectral properties, the kernel dimension and connected components, and the physical analogy with Newton's law of cooling / heat diffusion.
2. **Part 2: Spectral Community Detection & RatioCut** — Implement the 2-community (Fiedler vector) and $k$-community spectral clustering algorithms derived from the continuous relaxation of the RatioCut problem.
3. **Part 3: Stochastic Block Model (SBM) & Phase Transition** — Generate synthetic random graphs from the 2-class SBM $G(n, p, q)$ with logarithmic average degree ($p = \alpha \log(n)/n, q = \beta \log(n)/n$), investigate empirical recovery performance, and compare against the information-theoretic threshold $\sqrt{\alpha} - \sqrt{\beta} > \sqrt{2}$.
4. **Part 4: Matrix Lifting & Semidefinite Programming (SDP) Relaxation** — Formulate and solve the SDP relaxation with `cvxpy`, verify the theoretical **dual certificate** $Y = 	ext{diag}(2Ag \odot g - 1)$, and inspect the spectral gap $\lambda_2(Y - B) > 0$.
5. **Part 5: Application to Real-World Benchmark Datasets** — Test our spectral and modularity methods on benchmark networks:
   - **Zachary's Karate Club** (classic social network with ground truth split).
   - **Polblogs / College Football / Web Network** (real-world dataset fetched directly online).

---
### Setup & Dependencies
Ensure you have `numpy`, `scipy`, `matplotlib`, `networkx`, `scikit-learn`, and `cvxpy` installed:
```bash
pip install numpy scipy matplotlib networkx scikit-learn cvxpy
```



In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.linalg as la
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import cvxpy as cp
import urllib.request
import io
import tarfile

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 11



---
## Part 1: The Graph Laplacian & Spectral Connectivity

Let $G = (V, E)$ be an undirected unweighted graph with $n = |V|$ nodes and adjacency matrix $A \in \{0, 1\}^{n 	imes n}$ where $A_{ij} = 1$ if $(i,j) \in E$ and $0$ otherwise.
The degree matrix is $D = 	ext{diag}(d_1, \dots, d_n)$ where $d_i = \sum_{j=1}^n A_{ij}$.
The unnormalized graph Laplacian is defined as:
$$L = D - A$$

Recall the foundational quadratic form identity:
$$f^	op L f = \frac{1}{2} \sum_{(i,j) \in E} (f_i - f_j)^2 \ge 0, \quad \forall f \in \mathbb{R}^n$$
and the fact that $	ext{dim}(\ker L)$ equals the number of connected components of $G$.

---
### Question 1.1: Computing the Laplacian and Verifying the Kernel
1. Write a function `compute_laplacian(A)` returning the Laplacian matrix $L$ and the diagonal degree matrix $D$.
2. Generate a synthetic graph with 3 disjoint connected components:
   - Component 1: Clique $K_6$
   - Component 2: Cycle $C_8$
   - Component 3: Complete bipartite graph $K_{4,4}$
3. Compute the eigenvalues and eigenvectors of $L$. Verify numerically that:
   - $\lambda_1 = \lambda_2 = \lambda_3 = 0$.
   - The indicator vectors of the three components $\mathbf{1}_{V_1}, \mathbf{1}_{V_2}, \mathbf{1}_{V_3}$ span $\ker(L)$.



In [ ]:
# [ANSWER 1.1]
def compute_laplacian(A):
    """Compute degree matrix D and Laplacian L = D - A."""
    degrees = np.sum(A, axis=1)
    D = np.diag(degrees)
    L = D - A
    return L, D

# 1. Create the graph with 3 disjoint components
G1 = nx.complete_graph(6)             # V1: 6 nodes (0 to 5)
G2 = nx.cycle_graph(8)                # V2: 8 nodes (6 to 13)
G3 = nx.complete_bipartite_graph(4, 4)# V3: 8 nodes (14 to 21)

G_disjoint = nx.disjoint_union(nx.disjoint_union(G1, G2), G3)
n = G_disjoint.number_of_nodes()
A_disjoint = nx.to_numpy_array(G_disjoint)

# 2. Compute Laplacian and spectrum
L_disjoint, D_disjoint = compute_laplacian(A_disjoint)
eigvals, eigvecs = la.eigh(L_disjoint)

print(f"Total nodes: {n}")
print("First 6 smallest eigenvalues of L:")
for idx, val in enumerate(eigvals[:6]):
    print(f"  lambda_{idx+1} = {val:.6f}")

# Check that the first 3 eigenvalues are virtually 0
print(f"\nNumber of zero eigenvalues (numerical tol 1e-10): {np.sum(np.abs(eigvals) < 1e-10)}")

# Check span of Ker(L): eigenvectors corresponding to 0
ker_basis = eigvecs[:, :3]
# Component indicator vectors
ind1 = np.zeros(n); ind1[0:6] = 1.0; ind1 /= np.linalg.norm(ind1)
ind2 = np.zeros(n); ind2[6:14] = 1.0; ind2 /= np.linalg.norm(ind2)
ind3 = np.zeros(n); ind3[14:22] = 1.0; ind3 /= np.linalg.norm(ind3)
Ind_mat = np.column_stack([ind1, ind2, ind3])

# Projection of indicator vectors onto eigenspace of 0
proj = ker_basis @ (ker_basis.T @ Ind_mat)
reconstruction_error = np.linalg.norm(Ind_mat - proj)
print(f"Reconstruction error of true indicators in Ker(L): {reconstruction_error:.2e}")



---
### Question 1.2: Newton's Law of Cooling and Heat Diffusion on Graphs
The course notes explain that the name *Laplacian* arises from Newton's law of cooling:
$$\frac{d T}{dt} = -\kappa L T \implies T(t) = \exp(-\kappa L t) T(0)$$
where $T(t) \in \mathbb{R}^n$ is the temperature vector across the graph nodes at time $t$, and $\kappa > 0$ is the thermal conductivity coefficient.

1. Implement a simulation of heat diffusion starting from an initial temperature profile where:
   - Node 0 has $T_0(0) = 100^\circ\text{C}$,
   - All other nodes have $T_i(0) = 0^\circ\text{C}$.
2. Run this on a connected Barabási-Albert graph ($n=50$).
3. Plot the temperature evolution of a few selected nodes over time $t \in [0, 5]$.
4. What is the asymptotic equilibrium temperature $\lim_{t \to \infty} T(t)$ in terms of the initial vector $T(0)$ and the eigenvector $\mathbf{1}$? Verify your answer.



In [ ]:
# [ANSWER 1.2]
# Generate connected graph
G_heat = nx.barabasi_albert_graph(50, 3, seed=42)
A_heat = nx.to_numpy_array(G_heat)
L_heat, _ = compute_laplacian(A_heat)

# Initial temperature: node 0 is hot (100 C), others 0 C
n_heat = 50
T0 = np.zeros(n_heat)
T0[0] = 100.0

kappa = 0.5
time_steps = np.linspace(0, 5, 100)
T_traj = np.zeros((len(time_steps), n_heat))

# Compute matrix exponential trajectory T(t) = exp(-kappa * L * t) T0
for i, t in enumerate(time_steps):
    T_traj[i] = la.expm(-kappa * L_heat * t) @ T0

plt.figure(figsize=(9, 4.5))
plt.plot(time_steps, T_traj[:, 0], label="Node 0 (Source, initial 100°C)", lw=2, color='crimson')
neighbors_0 = list(G_heat.neighbors(0))[:3]
for nb in neighbors_0:
    plt.plot(time_steps, T_traj[:, nb], '--', label=f"Neighbor Node {nb}")
plt.plot(time_steps, T_traj[:, -1], ':', label="Distant Node 49", lw=2)

# Theoretical asymptotic temperature: projection onto Ker(L) = span(1)
T_inf_theoretical = np.mean(T0)  # (1^T T0 / n) * 1
plt.axhline(T_inf_theoretical, color='black', linestyle='-.', label=f"Equilibrium $T_\infty$ = {T_inf_theoretical:.1f}°C")

plt.xlabel("Time $t$")
plt.ylabel("Temperature (°C)")
plt.title("Heat Diffusion on Graph: $\frac{dT}{dt} = -\kappa L T$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Empirical average temperature at t=5: {np.mean(T_traj[-1]):.4f}°C")
print(f"Theoretical steady-state temperature:  {T_inf_theoretical:.4f}°C")



---
## Part 2: Spectral Community Detection & Continuous RatioCut Relaxation

Recall from Section 3.2 of the lecture notes:
For a 2-partition $V = V_1 \sqcup V_2$, the RatioCut is defined as:
$$\text{RatioCut}(V_1, V_2) = \frac{W(V_1, V_2)}{|V_1|} + \frac{W(V_2, V_1)}{|V_2|}$$
Defining the assignment vector $f \in \mathbb{R}^n$:
$$f_i = \begin{cases} \sqrt{\frac{|V_2|}{|V_1|}} & \text{if } i \in V_1 \\ -\sqrt{\frac{|V_1|}{|V_2|}} & \text{if } i \in V_2 \end{cases}$$
we have $\langle f, \mathbf{1}\rangle = 0$, $\|f\|_2^2 = n$, and:
$$f^\top L f = n \,\text{RatioCut}(V_1, V_2)$$

Relaxing the discrete set to continuous vectors $f \in \mathbb{R}^n$ yields:
$$\min_{f \in \mathbb{R}^n} f^\top L f \quad \text{s.t.} \quad \|f\|_2^2 = 1, \; \langle f, \mathbf{1} \rangle = 0$$
whose exact solution is given by the **Fiedler vector** $u_2$ (the eigenvector of $L$ associated with the second smallest eigenvalue $\lambda_2$, also called the *algebraic connectivity*).

For $k$ communities, Algorithm 1 from the lecture notes collects the $k$ eigenvectors $[u_1, \dots, u_k]$ of $L$ as columns of $H \in \mathbb{R}^{n \times k}$, and clusters the row vectors of $H$ using $k$-means.

---
### Question 2.1: Implementation of Spectral Clustering
1. Implement the function `spectral_clustering(A, k=2)` that:
   - Computes the Laplacian $L = D - A$.
   - Computes the $k$ eigenvectors associated with the $k$ smallest eigenvalues.
   - Normalizes rows if desired or clusters the rows using $k$-means.
   - Returns the community labels $\hat{y} \in \{0, \dots, k-1\}^n$ and the eigenvalues.
2. Test your algorithm on Zachary's Karate Club graph ($k=2$). Compare the detected communities with the ground truth factions (Mr. Hi vs the Officer) using the **Adjusted Rand Index (ARI)**.



In [ ]:
# [ANSWER 2.1]
def spectral_clustering(A, k=2, normalized=False):
    """
    Spectral Clustering Algorithm (Lecture Algorithm 1).
    Parameters:
      A: Adjacency matrix (n x n)
      k: Number of communities
      normalized: Whether to use the symmetric normalized Laplacian L_sym = D^{-1/2} L D^{-1/2}
    """
    n = A.shape[0]
    degrees = np.sum(A, axis=1)
    
    if not normalized:
        L = np.diag(degrees) - A
        eigvals, eigvecs = la.eigh(L)
        H = eigvecs[:, :k]
    else:
        # Ng-Jordan-Weiss / Shi-Malik normalized Laplacian variant
        d_inv_sqrt = np.power(np.maximum(degrees, 1e-12), -0.5)
        D_inv_sqrt = np.diag(d_inv_sqrt)
        L_sym = np.eye(n) - D_inv_sqrt @ A @ D_inv_sqrt
        eigvals, eigvecs = la.eigh(L_sym)
        H = eigvecs[:, :k]
        # Normalize rows to have unit Euclidean norm
        row_norms = np.linalg.norm(H, axis=1, keepdims=True)
        H = H / np.maximum(row_norms, 1e-12)

    # Cluster rows of H using k-means
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = kmeans.fit_predict(H)
    return labels, eigvals[:k+3], H

# Evaluate on Zachary's Karate Club
G_karate = nx.karate_club_graph()
A_karate = nx.to_numpy_array(G_karate)

# Ground truth split: 'Mr. Hi' (club: 'Mr. Hi') vs 'Officer' (club: 'Officer')
true_labels = np.array([0 if G_karate.nodes[i]['club'] == 'Mr. Hi' else 1 for i in G_karate.nodes()])

# Run Spectral Clustering (RatioCut relaxation)
pred_labels_unnorm, eigvals_karate, H_karate = spectral_clustering(A_karate, k=2, normalized=False)
pred_labels_norm, _, _ = spectral_clustering(A_karate, k=2, normalized=True)

ari_unnorm = adjusted_rand_score(true_labels, pred_labels_unnorm)
ari_norm = adjusted_rand_score(true_labels, pred_labels_norm)

print(f"Zachary's Karate Club (n={len(true_labels)}):")
print(f"  ARI (Unnormalized RatioCut Spectral): {ari_unnorm:.4f}")
print(f"  ARI (Normalized Spectral):           {ari_norm:.4f}")

# Visualize the Fiedler vector coordinates
fiedler_vec = H_karate[:, 1]
sorted_idx = np.argsort(fiedler_vec)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(range(len(fiedler_vec)), fiedler_vec[sorted_idx], c=true_labels[sorted_idx], cmap='coolwarm', s=50)
plt.axhline(0, color='gray', linestyle='--')
plt.title("Sorted Fiedler Vector Coordinates $u_2(i)$")
plt.xlabel("Sorted Node Index")
plt.ylabel("Value in $u_2$")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
pos = nx.spring_layout(G_karate, seed=42)
nx.draw_networkx_nodes(G_karate, pos, node_color=pred_labels_norm, cmap='coolwarm', node_size=250)
nx.draw_networkx_edges(G_karate, pos, alpha=0.4)
nx.draw_networkx_labels(G_karate, pos, font_size=8)
plt.title("Detected Communities (Karate Club)")
plt.axis('off')
plt.tight_layout()
plt.show()



---
## Part 3: The Stochastic Block Model (SBM) and the Detectability Phase Transition

Consider a graph $G \sim \text{SBM}(n, p, q)$ with two balanced communities $V_1, V_2$ of size $n/2$:
- Intra-community edge probability: $p$
- Inter-community edge probability: $q < p$

### The Exact Recovery Phase Transition (Theorem 4.3)
In the logarithmic sparsity regime:
$$p = \frac{\alpha \log n}{n}, \quad q = \frac{\beta \log n}{n}$$
- If $\sqrt{\alpha} - \sqrt{\beta} > \sqrt{2}$, exact recovery is information-theoretically possible.
- If $\sqrt{\alpha} - \sqrt{\beta} < \sqrt{2}$, no algorithm can recover the partition with probability $> 1/2$.

Moreover, the lecture proves:
- A spectral method on $A - \lambda J_n$ (Algorithm 2) achieves consistent recovery.
- An SDP relaxation reaches the exact phase transition threshold (Theorem 6.8).

---
### Question 3.1: Simulating SBM and the Leading Eigenvector of $A - \lambda J_n$
1. Write a function `generate_sbm_2blocks(n, p, q)` returning the adjacency matrix $A$ and true label vector $g \in \{-1, +1\}^n$.
2. Compute $\lambda = \frac{2}{n(n-1)} \sum_{i<j} A_{ij}$.
3. As shown in Section 4.2 of the lecture, find the leading eigenvector $\hat{x}$ of $M = A - \lambda J_n$ and estimate communities as $\text{sign}(\hat{x})$.
4. Evaluate recovery accuracy for various values of $(\alpha, \beta)$.



In [ ]:
# [ANSWER 3.1]
def generate_sbm_2blocks(n, p, q):
    """Generates a 2-class balanced SBM graph."""
    assert n % 2 == 0
    half = n // 2
    # g = +1 for V1, -1 for V2
    g = np.ones(n, dtype=int)
    g[half:] = -1
    
    # Generate upper triangular Bernoulli entries
    rand_mat = np.random.rand(n, n)
    # Probability matrix
    prob_mat = np.full((n, n), q)
    prob_mat[:half, :half] = p
    prob_mat[half:, half:] = p
    
    A_upper = (rand_mat < prob_mat).astype(float)
    A = np.triu(A_upper, 1)
    A = A + A.T
    np.fill_diagonal(A, 1.0) # diagonal set to 1 as in lecture convention
    return A, g

def spectral_sbm_recovery(A):
    """Spectral recovery via leading eigenvector of A - lambda * J_n (Section 4.2)."""
    n = A.shape[0]
    # lambda = 2 / (n(n-1)) * sum_{i < j} A_{ij}
    idx_upper = np.triu_indices(n, k=1)
    lam = np.mean(A[idx_upper])
    
    Jn = np.ones((n, n))
    M = A - lam * Jn
    
    # Leading eigenvector of M
    eigvals, eigvecs = la.eigh(M)
    x_hat = eigvecs[:, -1] # eigenvector for largest eigenvalue
    
    # Predicted community vector
    pred_g = np.sign(x_hat)
    pred_g[pred_g == 0] = 1 # handle rare exact 0
    return pred_g, x_hat

# Test on a realization
n_test = 200
alpha, beta = 16.0, 2.0
p_test = alpha * np.log(n_test) / n_test
q_test = beta * np.log(n_test) / n_test

A_sbm, g_true = generate_sbm_2blocks(n_test, p_test, q_test)
g_pred, x_hat = spectral_sbm_recovery(A_sbm)

# Alignment metric accounting for global sign flip
accuracy = max(np.mean(g_pred == g_true), np.mean(g_pred == -g_true))
print(f"SBM test (n={n_test}, alpha={alpha}, beta={beta}):")
print(f"  sqrt(alpha) - sqrt(beta) = {np.sqrt(alpha) - np.sqrt(beta):.3f} (Threshold = {np.sqrt(2):.3f})")
print(f"  Exact Recovery Accuracy: {accuracy * 100:.2f}%")



---
### Question 3.2: Empirical Investigation of the Phase Transition
Let $n = 300$, fix $\beta = 1.0$, and vary $\alpha \in [1.5, 9.0]$.
1. For each $\alpha$, run 10 Monte-Carlo trials and record the average exact recovery rate (proportion of trials where all nodes are correctly partitioned).
2. Plot the exact recovery probability as a function of $\sqrt{\alpha} - \sqrt{\beta}$.
3. Mark the theoretical threshold $\sqrt{2} \approx 1.414$ on the plot. Comment on the sharpness of the transition.



In [ ]:
# [ANSWER 3.2]
n_mc = 250
beta_val = 1.0
alpha_vals = np.linspace(1.2, 9.0, 15)
n_trials = 8

recovery_rates = []
gap_vals = []

for a in alpha_vals:
    p_val = a * np.log(n_mc) / n_mc
    q_val = beta_val * np.log(n_mc) / n_mc
    if p_val >= 1.0:
        continue
    
    successes = 0
    for _ in range(n_trials):
        A_mc, g_mc = generate_sbm_2blocks(n_mc, p_val, q_val)
        g_hat, _ = spectral_sbm_recovery(A_mc)
        # Check exact recovery (100% agreement up to sign)
        acc = max(np.mean(g_hat == g_mc), np.mean(g_hat == -g_mc))
        if acc >= 0.98: # near-perfect / exact recovery
            successes += 1
            
    recovery_rates.append(successes / n_trials)
    gap_vals.append(np.sqrt(a) - np.sqrt(beta_val))

plt.figure(figsize=(8, 4.5))
plt.plot(gap_vals, recovery_rates, 'o-', color='navy', lw=2, label="Empirical Recovery Rate (Spectral)")
plt.axvline(np.sqrt(2), color='red', linestyle='--', lw=2, label=r"Phase Transition: $\sqrt{\alpha} - \sqrt{\beta} = \sqrt{2}$")
plt.xlabel(r"$\sqrt{\alpha} - \sqrt{\beta}$")
plt.ylabel("Exact Recovery Probability")
plt.title(f"SBM Phase Transition for Exact Recovery ($n={n_mc}$)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



---
## Part 4: Matrix Lifting, SDP Relaxation, and Dual Certificate

From Section 6 of the course notes:
In the 2-community balanced SBM, the combinatorial maximum likelihood problem is:
$$\max_{x \in \{-1, +1\}^n, \langle x, \mathbf{1}\rangle = 0} x^\top A x$$
Adding the penalty $-\frac{1}{2}\langle x, \mathbf{1}\rangle^2 - \frac{1}{2}\|x\|_2^2$ leads to maximizing $x^\top B x = \langle B, x x^\top \rangle$ with:
$$B = 2A - \mathbf{1}\mathbf{1}^\top - I_n$$

Lifting $x \mapsto Z = x x^\top$ and relaxing the non-convex rank-1 constraint $\text{rank}(Z) = 1$ yields the **Semidefinite Program (SDP)** (Equation 44):
$$\max_{Z \in \mathbb{R}^{n \times n}} \langle B, Z \rangle \quad \text{s.t.} \quad Z \succeq 0, \; Z_{ii} = 1, \; \forall i = 1, \dots, n$$

### Dual Certificate (Proposition 6.5)
The candidate matrix $Z^* = g g^\top$ is the **unique** solution of the SDP if the candidate dual certificate:
$$Y = \text{diag}\left( \frac{2(Ag)_i}{g_i} - 1 \right)_{i=1}^n = \text{diag}(2(D^\sim - D^\nsim) - 1)$$
satisfies:
$$\lambda_2(Y - B) > 0$$

---
### Question 4.1: Solving the SDP Relaxation with `cvxpy`
1. Write a function `solve_sbm_sdp(B)` using `cvxpy` to solve the SDP relaxation:
   $$\max_{Z} \text{Tr}(B Z) \quad \text{s.t.} \quad Z \succeq 0, \; \text{diag}(Z) = 1$$
2. Test it on an SBM instance ($n=40$). Verify whether the solution $\hat{Z}$ is of rank 1 and matches $g g^\top$.



In [ ]:
# [ANSWER 4.1]
def solve_sbm_sdp(B):
    """Solves the SDP relaxation of 2-community SBM using cvxpy."""
    n = B.shape[0]
    Z = cp.Variable((n, n), symmetric=True)
    
    # Objective: maximize <B, Z> = Tr(B @ Z)
    objective = cp.Maximize(cp.trace(B @ Z))
    
    # Constraints: Z >= 0 (PSD) and Z_ii = 1
    constraints = [Z >> 0, cp.diag(Z) == 1]
    
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS, eps=1e-5) # SCS or CLARABEL are available open-source solvers
    
    return Z.value, prob.status

# Generate small SBM instance for fast SDP solving
n_sdp = 40
alpha_sdp, beta_sdp = 18.0, 1.0
p_sdp = alpha_sdp * np.log(n_sdp) / n_sdp
q_sdp = beta_sdp * np.log(n_sdp) / n_sdp

A_sdp, g_sdp = generate_sbm_2blocks(n_sdp, p_sdp, q_sdp)
Jn = np.ones((n_sdp, n_sdp))
B_sdp = 2 * A_sdp - Jn - np.eye(n_sdp)

Z_opt, status = solve_sbm_sdp(B_sdp)
print(f"SDP Solver Status: {status}")

# Check rank of Z_opt via singular values / eigenvalues
eigvals_Z = np.linalg.eigvalsh(Z_opt)
print(f"Top 3 eigenvalues of recovered Z_opt: {eigvals_Z[-3:][::-1]}")
print(f"Ratio lambda_2(Z) / lambda_1(Z): {eigvals_Z[-2] / eigvals_Z[-1]:.2e} (approx 0 confirms rank 1!)")

# Compare with true Z* = g g^T
Z_true = np.outer(g_sdp, g_sdp)
rel_error = np.linalg.norm(Z_opt - Z_true, 'fro') / np.linalg.norm(Z_true, 'fro')
print(f"Relative Frobenius error ||Z_opt - g g^T||_F / ||g g^T||_F: {rel_error:.4f}")



---
### Question 4.2: Constructing and Checking the Dual Certificate
1. Given $B$, $A$, and the ground-truth assignment $g$, compute the candidate dual certificate $Y \in \mathbb{R}^{n \times n}$:
   $$Y_{ii} = \frac{2(Ag)_i}{g_i} - 1, \quad Y_{ij} = 0 \; (i \ne j)$$
2. Compute the eigenvalues of $M_{dual} = Y - B$.
3. Check that:
   - The smallest eigenvalue $\lambda_1(Y - B) \approx 0$ with eigenvector $g$.
   - The second smallest eigenvalue $\lambda_2(Y - B) > 0$.
4. According to Proposition 6.5, does this certify that $Z^* = g g^\top$ is the unique optimal solution without having to run an SDP solver?



In [ ]:
# [ANSWER 4.2]
def verify_dual_certificate(A, B, g):
    """Constructs Y and checks the second smallest eigenvalue of Y - B."""
    n = len(g)
    # Y_ii = 2 * (A g)_i / g_i - 1
    Ag = A @ g
    y_diag = 2.0 * (Ag / g) - 1.0
    Y = np.diag(y_diag)
    
    # M_dual = Y - B
    M_dual = Y - B
    eigvals, eigvecs = la.eigh(M_dual)
    
    # Check that M_dual @ g = 0
    res_g = np.linalg.norm(M_dual @ g)
    
    return eigvals, eigvecs, res_g, Y

eigvals_dual, eigvecs_dual, res_g, Y = verify_dual_certificate(A_sdp, B_sdp, g_sdp)

print("Eigenvalues of Y - B:")
print(f"  lambda_1(Y - B) = {eigvals_dual[0]:.6f} (residual ||(Y-B)g|| = {res_g:.2e})")
print(f"  lambda_2(Y - B) = {eigvals_dual[1]:.6f}")
print(f"  lambda_3(Y - B) = {eigvals_dual[2]:.6f}")

if eigvals_dual[1] > 1e-4:
    print("\n--> CERTIFICATE VALID! lambda_2(Y - B) > 0.")
    print("    Proposition 6.5 rigorously guarantees that Z* = g g^T is the UNIQUE global minimizer of the SDP.")
else:
    print("\n--> Certificate condition failed (signal was too weak or graph not in exact recovery regime).")



## Part 5: Benchmark Datasets from the Web

We now evaluate our community detection algorithms on real networks:
1. **Network 1: Zachary's Karate Club** (built-in ground truth).
2. **Network 2: American College Football Network** (Girvan & Newman, 2002) or **Polblogs Network**.
   The Football network represents the schedule of Division I games for the 2000 season, where nodes are teams and edges represent regular season matches. Communities correspond to conferences (12 conferences).


### Question 5.1: Downloading and Loading a Real Graph from the Web
1. Download the American College Football network dataset or Political Blogs dataset from the web (or NetworkX dataset repository).
2. Parse the network into a `nx.Graph`, print basic statistics: number of nodes, number of edges, and ground-truth communities.
3. Compute the modularity and ARI achieved by $k$-way spectral clustering.



In [ ]:
# [ANSWER 5.1]
# Fetch the American College Football graph directly from web / NetworkX data repository
football_url = "http://www-personal.umich.edu/~mejn/netdata/football.zip"

try:
    import urllib.request
    import zipfile
    
    req = urllib.request.Request(football_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp:
        zip_data = zipfile.ZipFile(io.BytesIO(resp.read()))
        # Read the GML file from the zip
        gml_content = zip_data.read('football.gml').decode('utf-8')
        G_football = nx.parse_gml(gml_content)
        print("Successfully downloaded and parsed College Football network from UMich repository!")
except Exception as e:
    print(f"Network download encountered: {e}. Falling back to standard benchmark graph.")
    # Fallback: Florentine families or built-in Les Miserables graph
    G_football = nx.les_miserables_graph()

# Inspect graph
n_nodes = G_football.number_of_nodes()
n_edges = G_football.number_of_edges()
print(f"Graph Statistics: {n_nodes} nodes, {n_edges} edges")

# Extract ground-truth community (conference) if available
has_ground_truth = 'value' in list(G_football.nodes(data=True))[0][1]
if has_ground_truth:
    node_list = list(G_football.nodes())
    node_to_idx = {node: i for i, node in enumerate(node_list)}
    true_confs = [G_football.nodes[n]['value'] for n in node_list]
    k_football = len(set(true_confs))
    print(f"Number of ground-truth conferences / communities: {k_football}")
    
    A_fb = nx.to_numpy_array(G_football, nodelist=node_list)
    pred_fb, eigvals_fb, _ = spectral_clustering(A_fb, k=k_football, normalized=True)
    
    ari_fb = adjusted_rand_score(true_confs, pred_fb)
    nmi_fb = normalized_mutual_info_score(true_confs, pred_fb)
    print(f"\nSpectral Clustering on College Football Network:")
    print(f"  Adjusted Rand Index (ARI): {ari_fb:.4f}")
    print(f"  Normalized Mutual Info (NMI): {nmi_fb:.4f}")
else:
    A_fb = nx.to_numpy_array(G_football)
    pred_fb, eigvals_fb, _ = spectral_clustering(A_fb, k=5, normalized=True)
    print(f"Clustering into 5 communities completed.")

# Plot eigenvalue spectrum of the Laplacian to illustrate the 'elbow' heuristic
L_fb, _ = compute_laplacian(A_fb)
eigvals_all = np.sort(la.eigvalsh(L_fb))

plt.figure(figsize=(8, 4))
plt.plot(range(1, 25), eigvals_all[:24], 'o-', color='purple')
if has_ground_truth:
    plt.axvline(k_football, color='red', linestyle='--', label=f"True k = {k_football}")
plt.title("Spectrum of the Graph Laplacian (Smallest Eigenvalues)")
plt.xlabel("Index $i$")
plt.ylabel("Eigenvalue $\lambda_i(L)$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



---
## Summary of Key Takeaways
1. **Laplacian Spectrum & Connectivity:** The number of zero eigenvalues of $L = D - A$ directly counts the number of connected components $\ker(L) = \text{span}(\mathbf{1}_{V_1}, \dots, \mathbf{1}_{V_k})$.
2. **Continuous Relaxation of RatioCut:** Minimizing RatioCut is NP-hard, but relaxing the assignment vector $f$ to the sphere $\|f\|_2^2 = 1, f \perp \mathbf{1}$ yields the Fiedler vector $u_2$ as the exact optimum.
3. **Information-Theoretic vs Computational Limits:** In the logarithmic degree regime $p = \alpha \log n / n, q = \beta \log n / n$, the fundamental phase transition is $\sqrt{\alpha} - \sqrt{\beta} > \sqrt{2}$. Both spectral methods and SDP relaxations achieve this bound.
4. **SDP Relaxation & Dual Certification:** Matrix lifting $Z = x x^\top$ convexifies combinatorial cuts. The construction of the dual certificate $Y = \text{diag}(2Ag \odot g - 1)$ provides a verifiable condition $\lambda_2(Y - B) > 0$ guaranteeing that the SDP recovers the exact ground truth without any rank gap.

